# run_colab — thin launcher

All experiment logic lives in the `pidlora` package; this notebook only mounts storage, installs deps, and calls `python -m pidlora.train`. If a line of code affects a result, it does not belong in this notebook — put it in the package instead.

**Set `CONFIG_NAME` below, then Run All.** `--resume` is the default: after a Colab disconnect, re-running this notebook resumes from the last checkpoint on Drive automatically. Use `RESUME = False` only for a deliberate restart (this discards any existing checkpoint progress for this run).

Scope note: only the static-alpha branches (`baseline`, `sweep_a8`) exist right now.

In [ ]:
# Cell 1 — mount Drive + git clone/pull
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/dudesup/llm-pid-alignment.git"
REPO_DIR = "/content/llm-pid-alignment"
DRIVE_RUNS_DIR = "/content/drive/MyDrive/llm-pid-alignment/runs"  # survives session death — /content does not

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.makedirs(DRIVE_RUNS_DIR, exist_ok=True)
%cd {REPO_DIR}

In [ ]:
# Cell 2 — install dependencies
!pip install -q -r requirements.txt

In [ ]:
# Cell 3 — train (resume-by-default)
CONFIG_NAME = "baseline"   # one of: baseline | sweep_a8
RESUME = True               # False only for a deliberate restart

CONFIG_PATH = f"config/{CONFIG_NAME}.yaml"
OUTPUT_DIR = f"{DRIVE_RUNS_DIR}/{CONFIG_NAME}"

resume_flag = "--resume" if RESUME else "--no-resume"
!python -m pidlora.train --config {CONFIG_PATH} --output-dir {OUTPUT_DIR} {resume_flag}

In [ ]:
# Cell 4 — quick look: loss and KL so far (full plotting lives in analysis.ipynb)
import matplotlib.pyplot as plt
from pidlora.logging_utils import read_metrics_jsonl

records = read_metrics_jsonl(f"{OUTPUT_DIR}/metrics.jsonl")

train_steps = [r["step"] for r in records if r["event"] == "train_step"]
train_loss = [r["loss"] for r in records if r["event"] == "train_step"]
kl_steps = [r["step"] for r in records if r["event"] == "kl_eval"]
kl_raw = [r["kl_raw"] for r in records if r["event"] == "kl_eval"]
kl_filt = [r["kl_filt"] for r in records if r["event"] == "kl_eval"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(train_steps, train_loss)
axes[0].set_title("Training loss")
axes[0].set_xlabel("step")
axes[1].plot(kl_steps, kl_raw, label="raw", alpha=0.5)
axes[1].plot(kl_steps, kl_filt, label="EMA")
axes[1].set_title("KL (control set)")
axes[1].set_xlabel("step")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 5 (scratch) — interactive debugging only. Clear this cell's output/contents
# before committing; it is not part of the experiment.
